# Feature Engineering for Segment Classification — the measurements

Produces every number in `docs/feature-engineering-research.md`.

**What this notebook does not do.** It fits no transformer and tunes nothing. It
is TF-IDF plus a linear SVM, which gives a floor, not a prediction.

**It never touches the final test.** `docs/findings/02_splitting/02_finalization_20261009.md`
reserves the test split and says it has not been scored. Everything here is
trained on `train` and scored on `validation`.

## Before running

Two commands from Sebastian's message, in the repo root:

```bash
python -m catalogiq --mode integrated --output-dir data/processed/cleaned
python -m scripts.split_data --input data/processed/cleaned/training_candidate.csv --output-dir data/processed/splits
```

That produces `data/processed/cleaned/training_candidate.csv` and
`data/processed/splits/rule_assignments.csv`, which is what §0 loads.

## 0. Setup

The split assignments are the team's, not ours. They are joined onto the
candidate partition on the provenance key `dataset` + `source_sha256` +
`source_row`, with `validate="one_to_one"` so a silent fan-out cannot happen.

In [ ]:
import re, sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split, GroupShuffleSplit

pd.set_option("display.width", 170); pd.set_option("display.max_rows", 200)

import sklearn
print("pandas", pd.__version__, " scikit-learn", sklearn.__version__)
print("Every model here passes random_state=0. Without it liblinear shuffles")
print("internally and the same run gives numbers that differ by a few tenths of a")
print("point. If your library versions differ from the ones the doc was written")
print("against, small differences are still possible - compare the gaps between")
print("rows, not the digits.")

def project_root(start="."):
    p = Path(start).resolve()
    for _ in range(8):
        if (p/"data"/"provided").is_dir() or (p/".git").exists(): return p
        if p.parent == p: break
        p = p.parent
    return Path.cwd()

ROOT = project_root()
if (ROOT/"src").is_dir(): sys.path.insert(0, str(ROOT/"src"))
DATA = ROOT/"data"; PROC = DATA/"processed"
print("project root:", ROOT)

CLEANED = PROC/"cleaned"/"training_candidate.csv"
SPLITS  = PROC/"splits"/"rule_assignments.csv"
for p in (CLEANED, SPLITS):
    if not p.is_file():
        raise SystemExit(f"Missing {p}. Run the two commands in the cell above first.")

RD = dict(dtype="string", keep_default_na=False, low_memory=False)
KEY = ["dataset","source_sha256","source_row"]

# The agreed model input, taken from the team's own code rather than retyped.
from catalogiq.segment_transformer import build_product_texts, MODEL_FIELDS, ModelConfig
CFG = ModelConfig()
print("MODEL_FIELDS         :", MODEL_FIELDS)
print("sequence_length      :", CFG.sequence_length)
print("field_character_limits:", CFG.field_character_limits)

cand = pd.read_csv(CLEANED, **RD)
asg  = pd.read_csv(SPLITS,  **RD)
d = cand.merge(asg[KEY+["split","group_id","record_id"]], on=KEY,
               how="inner", validate="one_to_one")
print(f"\ncandidate {len(cand):,}   assignments {len(asg):,}   joined {len(d):,}")

blank = lambda s: s.str.strip().str.lower().isin(["", "null"])
d = d[~blank(d["Segment"])].copy()
d["y"] = d["Segment"].str.strip()

TARGETS = ["Mnfr","Brand","Platform","Segment","Sub-Segment","TargetAgeGroup"]
NUMERIC = ["ProductRating","ProductReviewsCount","XRatXRev","ReviewsCount"]
for c in list(MODEL_FIELDS)+["ProductCategory","Retailer","ProductUrl"]:
    d[c] = d[c].str.strip().replace({"null":""})
d["cat"] = d["ProductCategory"].map(lambda s: re.sub(r"\s*>\s*", " > ", s))

TR = d[d["split"]=="train"]
VA = d[d["split"]=="validation"]
print(f"rows with a Segment label: {len(d):,}")
print(f"  train      {len(TR):,}")
print(f"  validation {len(VA):,}")
print(f"  test       {(d['split']=='test').sum():,}  <- reserved, never scored here")

## 1. Coverage and class support

Two of the four allowlisted fields are the ones to look at.

In [ ]:
rows = []
for c in ["ProductName","Retailer","ProductUrl","ProductRating","XRatXRev","ReviewsCount",
          "ProductCategory","ProductBrand","ProductReviewsCount","ProductImageUrl",
          "ProductContents","ProductDescription"]:
    if c not in d.columns: continue
    rows.append({"column": c, "populated_%": round(100*(1-blank(d[c]).mean()), 2),
                 "allowlisted": c in MODEL_FIELDS,
                 "distinct": d[c].str.strip().nunique()})
print(pd.DataFrame(rows).to_string(index=False))

print("\nSegment class support:")
vc = d["y"].value_counts()
print(pd.DataFrame({"rows": vc, "share_%": (vc/len(d)*100).round(1)}).to_string())
print(f"\nmajority-class baseline: {vc.iloc[0]/len(d)*100:.2f}%  ({vc.index[0]})")
print("Accuracy alone is therefore not a result. Macro F1 is the metric that moves.")

## 2. How much of Segment is a lookup table?

For a categorical column, map each value to its most common Segment and learn
nothing else. This measures how much of the target is memorisation rather than
understanding — useful both as a signal estimate and as a leakage warning.

Watch two columns together: **accuracy** and **keys**. High accuracy with few
keys is signal. High purity with many keys is memorisation.

In [ ]:
def lookup_ceiling(series, name, out):
    s = series.str.strip()
    t = pd.DataFrame({"k": s, "y": d["y"]})
    t = t[~blank(series)]
    if not len(t): return
    hit = t.groupby("k")["y"].agg(lambda g: g.value_counts().iloc[0]).sum()
    out.append({"feature as a lookup table": name,
                "keys": t["k"].nunique(),
                "rows_per_key": round(len(t)/t["k"].nunique(), 1),
                "accuracy_%": round(hit/len(t)*100, 2),
                "one_segment_keys_%": round((t.groupby("k")["y"].nunique()==1).mean()*100, 1)})

d["cat_norm"] = d["ProductCategory"].map(lambda s: re.sub(r"\s*>\s*", ">", s))
for lv in range(1, 5):
    d[f"cat_l{lv}"] = d["cat_norm"].map(lambda s, l=lv: ">".join(s.split(">")[:l]))
d["ret_cat"] = d["Retailer"] + " | " + d["cat_norm"]
d["url_dir"] = d["ProductUrl"].map(
    lambda u: "/".join(re.sub(r"^https?://[^/]+", "", u).split("?")[0].strip("/").split("/")[:-1]))

out = []
lookup_ceiling(d["Retailer"], "Retailer", out)
for lv in range(1, 5):
    lookup_ceiling(d[f"cat_l{lv}"], f"ProductCategory, first {lv} level(s)", out)
lookup_ceiling(d["cat_norm"], "ProductCategory, full path", out)
lookup_ceiling(d["ret_cat"],  "Retailer + full path", out)
lookup_ceiling(d["ProductBrand"], "ProductBrand", out)
lookup_ceiling(d["url_dir"], "ProductUrl directory path", out)
print(pd.DataFrame(out).to_string(index=False))

print("\nRead it like this:")
print("  Retailer       - accuracy equals the majority baseline, so it carries no signal.")
print("  Category path  - 93% from ~1,300 keys. Real signal.")
print("  ProductUrl     - 99.7% of keys are pure, but there are 10k+ keys over 79k rows.")
print("                   That is memorisation; its accuracy is barely above baseline.")

## 3. The feature-set comparison, on the official splits

Trained on `train`, scored on `validation`. The first row is the team's agreed
input, built by their own `build_product_texts` so there is no risk of my
version of it differing.

In [ ]:
d["agreed"]          = build_product_texts(d)
d["name_only"]       = "[ProductName] " + d["ProductName"]
d["name_cat"]        = d["name_only"] + " [ProductCategory] " + d["cat"]
d["agreed_plus_cat"] = d["agreed"] + " [ProductCategory] " + d["cat"]

TR = d[d["split"]=="train"]; VA = d[d["split"]=="validation"]

def text_model(col, train, test, ng=(1,2), mx=80000):
    ct = ColumnTransformer([("t", TfidfVectorizer(max_features=mx, ngram_range=ng,
                                                  min_df=2, sublinear_tf=True), col)],
                           sparse_threshold=1.0)
    clf = make_pipeline(ct, LinearSVC(C=1.0, class_weight="balanced", random_state=0,
                                      max_iter=3000, dual=True))
    clf.fit(train, train["y"])
    p = clf.predict(test)
    return accuracy_score(test["y"], p), f1_score(test["y"], p, average="macro"), p

INPUTS = [("the agreed four fields",              "agreed"),
          ("ProductName only",                    "name_only"),
          ("ProductName + ProductCategory",       "name_cat"),
          ("the agreed four fields + ProductCategory", "agreed_plus_cat")]

base = VA["y"].value_counts().iloc[0]/len(VA)
res, preds = [{"input":"majority class","accuracy_%":round(base*100,2),"macro_F1_%":None}], {}
for nm, col in INPUTS:
    a, f, p = text_model(col, TR, VA); preds[nm] = p
    res.append({"input": nm, "accuracy_%": round(a*100,2), "macro_F1_%": round(f*100,2)})
TABLE = pd.DataFrame(res)
print("TF-IDF word 1-2 grams + linear SVM, scored on validation")
print(TABLE.to_string(index=False))

g = TABLE.set_index("input")["macro_F1_%"]
print(f"\nAdding ProductCategory to the agreed input is worth "
      f"{g['the agreed four fields + ProductCategory'] - g['the agreed four fields']:+.2f}pp macro F1.")
print(f"The three non-name allowlisted fields are worth "
      f"{g['the agreed four fields'] - g['ProductName only']:+.2f}pp over ProductName alone.")

### 3.1 Per class

Where the category path actually helps: the minority Segments, which is where
macro F1 lives.

In [ ]:
a = classification_report(VA["y"], preds["the agreed four fields"],
                          output_dict=True, zero_division=0)
b = classification_report(VA["y"], preds["the agreed four fields + ProductCategory"],
                          output_dict=True, zero_division=0)
rows = []
for k in sorted(set(VA["y"])):
    rows.append({"Segment": k, "support": int(a[k]["support"]),
                 "F1_agreed": round(a[k]["f1-score"], 3),
                 "F1_with_category": round(b[k]["f1-score"], 3),
                 "gain_pp": round((b[k]["f1-score"]-a[k]["f1-score"])*100, 1)})
print(pd.DataFrame(rows).sort_values("support").to_string(index=False))

## 4. The cross-retailer test — why §3 is not the whole story

Amazon is most of the data. Train on Amazon alone and score the other five
retailers. This is the question "what happens on a retailer we have not seen",
and it is not one of the three official splits.

First the fact that drives the result: how many category paths do the retailers
share?

In [ ]:
am = d[d["Retailer"]=="Amazon"]; ot = d[d["Retailer"]!="Amazon"]
print(f"Amazon rows {len(am):,}   other five {len(ot):,}")
print(f"distinct category paths - Amazon {am['cat'].nunique():,}, others {ot['cat'].nunique():,}")
shared = len(set(ot["cat"]) & set(am["cat"]))
print(f"paths in the other five that also appear in Amazon: {shared} "
      f"of {ot['cat'].nunique():,}")
print()
rows = []
for nm, col in INPUTS:
    a_, f_, _ = text_model(col, am, ot)
    rows.append({"input": nm, "accuracy_%": round(a_*100,2), "macro_F1_%": round(f_*100,2)})
X = pd.DataFrame(rows)
print("trained on Amazon, scored on Walmart / Walgreens / Target / CVS / Kroger")
print(X.to_string(index=False))

h = X.set_index("input")["macro_F1_%"]
print(f"\nIn-distribution, ProductCategory is worth "
      f"{g['the agreed four fields + ProductCategory'] - g['the agreed four fields']:+.2f}pp macro F1.")
print(f"Cross-retailer, it is worth "
      f"{h['the agreed four fields + ProductCategory'] - h['the agreed four fields']:+.2f}pp.")
print("Same column, opposite sign. Which number matters is a product question.")

### 4.1 The path still works as *text* on unseen paths

Split by category path so that every scored row has a path never seen in
training. If the path were only a lookup key, accuracy would collapse. It does
not, because the path's words generalise even when the whole path is new.

**This is the argument for tokenising the path and never one-hot encoding it.**

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
itr, ite = next(gss.split(d, groups=d["cat"]))
ctr, cte = d.iloc[itr], d.iloc[ite]
print(f"train {len(ctr):,}  test {len(cte):,}")
print(f"scored rows whose category path is unseen in training: "
      f"{(~cte['cat'].isin(set(ctr['cat']))).mean()*100:.1f}%")
print()
rows = []
for nm, col in INPUTS:
    a_, f_, _ = text_model(col, ctr, cte)
    rows.append({"input": nm, "accuracy_%": round(a_*100,2), "macro_F1_%": round(f_*100,2)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nAccuracy holds up; macro F1 is what suffers. The minority classes are")
print("the ones that depended on having seen the exact path.")

### 4.2 How the path is represented changes the sign of the answer

§4 concatenates the category path into one text field, which is what
`build_product_texts` does. An obvious alternative is to give the path its own
TF-IDF block in a `ColumnTransformer`, so it gets a feature space of its own.

In-distribution the two are close. Cross-retailer they are not, and that is the
finding: **a separate block lets Amazon's taxonomy vocabulary dominate, and it
stops helping.** Concatenated, the path's tokens are diluted among the name's
and the model keeps leaning on the name.

In [ ]:
def blocks_model(specs, train, test):
    ct = ColumnTransformer([(n,t,c) for n,t,c in specs], sparse_threshold=1.0)
    clf = make_pipeline(ct, LinearSVC(C=1.0, class_weight="balanced", random_state=0,
                                      max_iter=3000, dual=True))
    clf.fit(train, train["y"]); p = clf.predict(test)
    return accuracy_score(test["y"], p), f1_score(test["y"], p, average="macro")

TF = lambda mx, ng: TfidfVectorizer(max_features=mx, ngram_range=ng, min_df=2, sublinear_tf=True)
SEPARATE = [("name", TF(80000,(1,2)), "ProductName"), ("cat", TF(80000,(1,3)), "cat")]

rows = []
for label, train, test in [("in-distribution (official splits)", TR, VA),
                           ("cross-retailer (Amazon -> other five)", am, ot)]:
    a1, f1_, _ = text_model("name_cat", train, test)              # concatenated
    a2, f2_    = blocks_model(SEPARATE, train, test)              # separate blocks
    a3, f3_, _ = text_model("name_only", train, test)             # no category at all
    rows.append({"evaluation": label,
                 "name only_F1": round(f3_*100,2),
                 "concatenated_F1": round(f1_*100,2),
                 "separate block_F1": round(f2_*100,2)})
R = pd.DataFrame(rows)
print(R.to_string(index=False))
print()
print("Read the cross-retailer row. Including the path helps when it is concatenated")
print("and stops helping when it gets its own block. Same column, same data, different")
print("representation. This is experiment E3 in the doc: how the path enters the model")
print("matters as much as whether it does.")

## 5. Columns with no signal

Worth running once so the experiment list can rule them out with a number
instead of an opinion. Scored on validation, same splits.

In [ ]:
for c in NUMERIC:
    d[c+"_n"] = pd.to_numeric(d[c], errors="coerce").fillna(-1)
d["name_len"] = d["ProductName"].str.len()
NUMS = [c+"_n" for c in NUMERIC] + ["name_len"]
TR = d[d["split"]=="train"]; VA = d[d["split"]=="validation"]

def generic(specs, train, test):
    ct = ColumnTransformer([(n,t,c) for n,t,c in specs], sparse_threshold=1.0)
    clf = make_pipeline(ct, LinearSVC(C=1.0, class_weight="balanced", random_state=0,
                                      max_iter=3000, dual=True))
    clf.fit(train, train["y"]); p = clf.predict(test)
    return accuracy_score(test["y"], p), f1_score(test["y"], p, average="macro")

rows = []
for nm, specs in [
    ("numerics + name length only", [("num", StandardScaler(), NUMS)]),
    ("Retailer only", [("ret", TfidfVectorizer(analyzer="word"), "Retailer")]),
]:
    a_, f_ = generic(specs, TR, VA)
    rows.append({"input": nm, "accuracy_%": round(a_*100,2), "macro_F1_%": round(f_*100,2)})
rows.append({"input": "majority class", "accuracy_%": round(base*100,2), "macro_F1_%": None})
print(pd.DataFrame(rows).to_string(index=False))
print("\nBoth are below the majority baseline on accuracy and near-zero on macro F1.")
print("A rating of 4.5 is a rating of 4.5 whether the product is an antacid or a")
print("multivitamin. Drop these from the Segment experiment list.")

## 6. The 128-token window

`sequence_length` is 128 tokens, roughly 400–550 characters of product text,
while `field_character_limits` allows 7,536 characters in. The question is which
fields actually reach the model.

In [ ]:
txt = pd.Series(build_product_texts(d), index=d.index)
print("example built input, first 240 characters:")
print(" ", repr(txt.iloc[0][:240]))
print()
print("built-text length in characters:")
print(txt.str.len().describe(percentiles=[.25,.5,.75,.9,.95]).round(0).to_string())
print()
rows = []
for f in MODEL_FIELDS:
    pos = txt.str.find("["+f+"]"); pr = pos>=0
    rows.append({"field": f, "present_%": round(pr.mean()*100,2),
                 "median_start_char": int(pos[pr].median()) if pr.any() else None})
print(pd.DataFrame(rows).to_string(index=False))
print()
print(f"share of rows whose built text exceeds the window:")
for cap in (400, 500, 550):
    print(f"  > {cap} chars (~128 tokens): {(txt.str.len()>cap).mean()*100:5.1f}%")
print()
print("ProductContents is allowlisted, present on about 38% of rows, and typically")
print("starts past the window. On most of the rows that have it, the model never")
print("sees it. Experiment E2 in the doc is about this.")

## 7. Does the grouping change the score?

A correction to something I wrote earlier. §2.7 of `model-evaluation-strategy.md`
said the absence of a grouping key "blocks the validity of every reported
number". This measures it with the key I actually recommended there: normalised
`ProductName` with size and count tokens stripped, plus `ProductBrand`.

In [ ]:
def norm_name(s):
    s = s.lower()
    s = re.sub(r"\b\d+(\.\d+)?\s*(ct|count|pk|pack|oz|fl|ml|l|g|mg|mcg|kg|lb|lbs"
               r"|tablets?|caps?|capsules?|softgels?|gummies|servings?|piece|pcs)\b", " ", s)
    s = re.sub(r"[^a-z ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

d["my_grp"] = d["ProductName"].map(norm_name) + " || " + d["ProductBrand"].str.lower()
sizes = d["my_grp"].value_counts()
print(f"rows {len(d):,}   groups {d['my_grp'].nunique():,}   "
      f"rows per group {len(d)/d['my_grp'].nunique():.2f}")
print(f"groups with more than one row: {(sizes>1).sum():,} "
      f"({(sizes>1).mean()*100:.1f}% of groups, {sizes[sizes>1].sum()/len(d)*100:.1f}% of rows)")
print()
tr_r, te_r = train_test_split(d, test_size=0.2, random_state=0, stratify=d["y"])
itr, ite = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
                .split(d, groups=d["my_grp"]))
tr_g, te_g = d.iloc[itr], d.iloc[ite]
rows = []
for nm, col in INPUTS:
    a1, f1, _ = text_model(col, tr_r, te_r)
    a2, f2, _ = text_model(col, tr_g, te_g)
    rows.append({"input": nm, "random_F1_%": round(f1*100,2),
                 "grouped_F1_%": round(f2*100,2), "drop_pp": round((f1-f2)*100,2)})
print("my own grouping key: random split against grouped split")
print(pd.DataFrame(rows).to_string(index=False))
print()
print("The gap is noise. Either near-duplicates are rarer in this catalogue than")
print("I assumed in 2.7, or that key is too strict to find them. The team's rule-v3")
print("grouping is more sophisticated and may catch what this misses - but the")
print("measured effect on Segment scores is the number to check.")

## 8. The official splits: what the validation set looks like

Where the real generalisation risk sits, and one issue worth sending back to
whoever owns the splitter.

In [ ]:
TR = d[d["split"]=="train"]; VA = d[d["split"]=="validation"]
print(f"validation rows whose group_id appears in train     : "
      f"{VA['group_id'].isin(set(TR['group_id'])).mean()*100:5.1f}%   (isolation holds)")
print(f"validation rows whose category path is unseen in train: "
      f"{(~VA['cat'].isin(set(TR['cat']))).mean()*100:5.1f}%")
print(f"non-Amazon rows whose category path is unseen in Amazon: "
      f"{(~ot['cat'].isin(set(am['cat']))).mean()*100:5.1f}%")
print()
rows = []
for ret in d["Retailer"].value_counts().index:
    rows.append({"Retailer": ret,
                 "train_%": round((TR["Retailer"]==ret).mean()*100, 1),
                 "validation_%": round((VA["Retailer"]==ret).mean()*100, 1)})
rt = pd.DataFrame(rows)
rt["shift_pp"] = (rt["validation_%"] - rt["train_%"]).round(1)
print(rt.to_string(index=False))
print()
print("Group isolation is clean. But validation is more Amazon-heavy than train,")
print("because groups are correlated with retailer. Every validation number leans")
print("that way. Stratifying the allocation by retailer as well as grouping would")
print("fix it, if that can be done without breaking isolation.")

## 9. Summary

In [ ]:
out = DATA/"interim"; out.mkdir(parents=True, exist_ok=True)
TABLE.to_csv(out/"segment_feature_comparison.csv", index=False)
print("written to", out/"segment_feature_comparison.csv")
print("  (data/interim/ is gitignored, so this file is not in the repo)")
print()
print(TABLE.to_string(index=False))
print()
print("Reminders for anyone reading a number off this notebook:")
print("  - validation only. the final test is reserved and was never scored here.")
print("  - TF-IDF and a linear SVM. a floor, not a prediction for the transformer.")
print("  - nothing is tuned. C=1.0 and defaults throughout.")
print("  - Segment only. the other five targets may rank features differently.")